# Task 2: Original LCS System on raw Dataset

## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from skeLCS import eLCS
import time

## Loading the raw dataset

In [2]:
#Load the raw dataset for the LCS baseline
df_raw_lcs = pd.read_csv('Yaacoub_creditcard.csv')
df_raw_lcs.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Minimal processing + stratified subsample

In [3]:
#Minimal processing needed for LCS compatibility
#separate features/target, take a stratified subsample for computational feasibility
X = df_raw_lcs.drop(columns=['Class']).values
y = df_raw_lcs['Class'].values

_, X_sample, _, y_sample = train_test_split(X, y, test_size=15000, stratify=y, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, stratify=y_sample, random_state=42)

print(f"Training set: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Test set: {X_test.shape}, Fraud cases: {y_test.sum()}")

Training set: (12000, 30), Fraud cases: 21
Test set: (3000, 30), Fraud cases: 5


## Run the original, unmodified eLCS baseline

In [4]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

preds = model.predict(X_test)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

Training time: 42.50 seconds
Accuracy: 0.9983
Balanced Accuracy: 0.5000
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000


### Task 2: Original LCS System on Raw Dataset (Results)
An original, unmodified eLCS system (scikit-eLCS implementation) was run on the raw dataset prior to any cleaning or transformation. Due to the computational cost of LCS rule evolution on the full 284,807-row dataset, a stratified subsample of 15,000 rows was used, preserving the original ~0.17% fraud rate (21 fraud cases in the 12,000-row training set; 5 in the 3,000-row test set). This subsampling, along with separating features and the target label into NumPy arrays, was the only processing applied. No cleaning, scaling, or transformation was done, in line with the requirement to establish a baseline on raw data. 

The model was trained using eLCS's default parameters (learning_iterations = 10,000, N = 1,000, p_spec = 0.5, nu = 5, chi = 0.8, mu = 0.04, theta_GA = 25) with no modification. Training took 42.50 seconds.  

**Baseline results:**  
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9983  |  
| Balanced Accuracy | 0.5000 |  
|Precision | 0.0000  |  
| Recall   | 0.0000  |  
| F1-score | 0.0000  |  

Although accuracy is very high, its misleading given the class imbalance. A classifier that predicts legitimate for every transaction would have similar accuracy. The balanced accuracy of 0.5, alongside zero precision, recall, and F1-score, shows the original LCS system failed to identify any fraudulent transactions in the test set, consistent with the small number of fraud examples in the raw, unprocessed training data (21 cases). This suggests the system didn't encounter enough fraud examples to evolve rules capable of distinguishing fraud from legitimate transactions. This result supports the preprocessing and improvement work done in Tasks 3 and 4.